# 06 — Serie temporali I: previsioni di riferimento, autocorrelazione, AR (1927–1938)

## Usage e contratti numerici
I kernel numerici sono generici IEEE; i casi seguenti usano double e non certificano ogni formato. Mean/Naive/SeasonalNaive/Drift sono baseline, non stimatori universali. Periodi, orizzonti e lag devono essere compatibili con la storia; drift richiede abbastanza osservazioni. ACF normalizza l'autocovarianza e puo' degenerare sulle serie costanti. Yule-Walker/Levinson richiedono un problema numericamente valido; Ljung-Box usa una distribuzione chi-quadro approssimata. Valutare preprocessing e fit solo sul passato; confrontare baseline sulle stesse finestre.
[Catalogo API](12-api-reference.ipynb), [verifiche sui sei formati](11-consolidation.ipynb), [limiti e copertura](../../ARCHITECTURE.md).
Kernel **JupyterNet** (C#) su runtime **.NET 11**. Prerequisito: `dotnet build src/MathemaSharp.slnx -c Release`. Da terminale: `scripts/run-notebooks.ps1` (imposta il roll-forward al runtime 11); in VS Code serve `DOTNET_ROLL_FORWARD=LatestMajor` e, finché .NET 11 è RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

Percorso in quattro notebook che segue la cronologia di `refs/timeseries/time-series-forecasting-01.jpg` — *una cronologia di idee, non una classifica di modelli*:

| Anni | Idea | Notebook | API `MathemaSharp.TimeSeries` |
|---|---|---|---|
| — | Mean, naive, seasonal naive | 06 | `ReferenceForecasts`, `Accuracy` |
| 1927–1938 | Yule (AR), Wold | 06 | `SeriesStatistics`, `ArModel` |
| 1950s–1960 | SES, Holt, Holt-Winters | 07 | `ExponentialSmoothing` |
| 1960 | Kalman | 07 | `KalmanFilter` |
| 1985 | Damped trend | 07 | `ExponentialSmoothing.Holt(..., phi)` |
| 1970 | Box-Jenkins / ARIMA | 08 | `ArimaModel` |
| 1980 | VAR | 08 | `VarModel` |
| 1982/1986 | ARCH/GARCH | 08 | `Garch11` |
| 1990 | STL | 09 | `Stl` |

In [ ]:
#load "../shared/MathemaSharp.PlotSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
using MathemaSharp.TimeSeries;
// Serie mensile sintetica: livello, trend lineare, stagionalità annuale, rumore gaussiano. Seme fisso: risultati riproducibili.
var random = new Random(2026);
const int period = 12;
var y = Enumerable.Range(0, 120).Select(t => 100 + 0.6 * t + 8 * Math.Sin(2 * Math.PI * t / period) + 3 * Simulation.StandardNormal(random)).ToArray();
double[] train = y[..108], test = y[108..];
Chart.Lines("Serie mensile sintetica (120 mesi)", new Line("y", y));

Serie mensile sintetica (120 mesi) 80 100 120 140 160 180 0 25 50 75 100 125 <path d='M64 178.2L69.06 159.3L74.11 154.82L79.17 157.3L84.22 166.95L89.28 162.65L94.34 176.74L99.39 178.2L104.45 176.67L109.5 180.28L114.56 178.48L119.62 171.67L124.67 168.76L129.73 149.55L134.78 152.95L139.84 148.05L144.9 150.8L149.95 156.4L155.01 156.98L160.06 167.51L165.12 177.57L170.18 174L175.23 165.4L180.29 159.47L185.34 152.36L190.4 142.32L195.46 141.02L200.51 138.98L205.57 131.35L210.62 145.43L215.68 150.54L220.74 148.2L225.79 156.81L230.85 160.72L235.9 160.56L240.96 151.3L246.02 139.6L251.07 145.29L256.13 125.43L261.18 128.58L266.24 138.09L271.3 124.98L276.35 140.69L281.41 142.13L286.46 141.46L291.52 142.73L296.58 139.49L301.63 137.85L306.69 139.16L311.74 124.68L316.8 109.36L321.86 123.34L326.91 124.44L331.97 119.06L337.02 128.91L342.08 123.77L347.14 138.95L352.19 137.9L357.25 139.89L362.3 116.74L367.36 126.23L372.42 117.71L377.47 98.47L382.53 102.79L387.58 104.44L392.64 106.5L397.7 118.78L402.75 113.15L407.81 131.19L412.86 126.44L417.92 120.78L422.98 120.42L428.03 109.47L433.09 92.96L438.14 101.19L443.2 94.86L448.26 96.94L453.31 99.88L458.37 99.59L463.42 100.97L468.48 109.91L473.54 118.71L478.59 118.63L483.65 97.79L488.7 100.83L493.76 90.73L498.82 85.7L503.87 86.77L508.93 78.26L513.98 84.21L519.04 88.43L524.1 96.5L529.15 98.15L534.21 94.56L539.26 92.86L544.32 94.32L549.38 78.81L554.43 73.57L559.49 66.53L564.54 69.01L569.6 68.34L574.66 82.22L579.71 75.98L584.77 83.51L589.82 82.07L594.88 84.34L599.94 91.97L604.99 76.33L610.05 70.67L615.1 66.48L620.16 64.73L625.22 56.48L630.27 51.75L635.33 59.42L640.38 69.05L645.44 63.11L650.5 78.07L655.55 77.79L660.61 73.9L665.66 67.01' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/> y · x=0: 99.6 y · x=1: 111 y · x=2: 114 y · x=3: 113 y · x=4: 107 y · x=5: 109 y · x=6: 101 y · x=7: 99.6 y · x=8: 101 y · x=9: 98.3 y · x=10: 99.5 y · x=11: 104 y · x=12: 105 y · x=13: 117 y · x=14: 115 y · x=15: 118 y · x=16: 117 y · x=17: 113 y · x=18: 113 y · x=19: 106 y · x=20: 100 y · x=21: 102 y · x=22: 108 y · x=23: 111 y · x=24: 116 y · x=25: 122 y · x=26: 123 y · x=27: 124 y · x=28: 129 y · x=29: 120 y · x=30: 117 y · x=31: 118 y · x=32: 113 y · x=33: 110 y · x=34: 111 y · x=35: 116 y · x=36: 123 y · x=37: 120 y · x=38: 132 y · x=39: 130 y · x=40: 124 y · x=41: 132 y · x=42: 123 y · x=43: 122 y · x=44: 122 y · x=45: 122 y · x=46: 124 y · x=47: 125 y · x=48: 124 y · x=49: 133 y · x=50: 142 y · x=51: 133 y · x=52: 133 y · x=53: 136 y · x=54: 130 y · x=55: 133 y · x=56: 124 y · x=57: 125 y · x=58: 123 y · x=59: 138 y · x=60: 132 y · x=61: 137 y · x=62: 149 y · x=63: 146 y · x=64: 145 y · x=65: 144 y · x=66: 136 y · x=67: 140 y · x=68: 129 y · x=69: 132 y · x=70: 135 y · x=71: 135 y · x=72: 142 y · x=73: 152 y · x=74: 147 y · x=75: 151 y · x=76: 150 y · x=77: 148 y · x=78: 148 y · x=79: 147 y · x=80: 142 y · x=81: 136 y · x=82: 136 y · x=83: 149 y · x=84: 147 y · x=85: 154 y · x=86: 157 y · x=87: 156 y · x=88: 161 y · x=89: 158 y · x=90: 155 y · x=91: 150 y · x=92: 149 y · x=93: 151 y · x=94: 152 y · x=95: 151 y · x=96: 161 y · x=97: 164 y · x=98: 169 y · x=99: 167 y · x=100: 167 y · x=101: 159 y · x=102: 163 y · x=103: 158 y · x=104: 159 y · x=105: 158 y · x=106: 153 y · x=107: 163 y · x=108: 166 y · x=109: 169 y · x=110: 170 y · x=111: 175 y · x=112: 178 y · x=113: 173 y · x=114: 167 y · x=115: 171 y · x=116: 161 y · x=117: 162 y · x=118: 164 y · x=119: 168

## Previsioni di riferimento
Sono ancora i benchmark di oggi: un modello che non le batte non giustifica la sua complessità.

- **Mean**: $\hat y_{T+h} = \bar y$ — ottima se la serie è rumore bianco attorno a una media.
- **Naive**: $\hat y_{T+h} = y_T$ — ottima per una passeggiata aleatoria.
- **Seasonal naive**: $\hat y_{T+h} = y_{T+h-m(k+1)}$, $k=\lfloor (h-1)/m \rfloor$.
- **Drift**: $\hat y_{T+h} = y_T + h\,\frac{y_T-y_1}{T-1}$.

Per confrontarle si usa la **MASE** (Hyndman & Koehler 2006): MAE fuori campione diviso il MAE in-sample del seasonal naive. È indipendente dalla scala, definita con gli zeri; MASE < 1 significa battere il naive stagionale *in-sample*.

In [ ]:
var forecasts = new (string Name, double[] F)[]
{
    ("Mean", ReferenceForecasts.Mean(train, 12)),
    ("Naive", ReferenceForecasts.Naive(train, 12)),
    ("Seasonal naive", ReferenceForecasts.SeasonalNaive(train, period, 12)),
    ("Drift", ReferenceForecasts.Drift(train, 12)),
};
Table.Show(new[] { "Metodo", "MAE", "RMSE", "MAPE %", "MASE (m=12)" },
    forecasts.Select(f => new object[] { f.Name, Accuracy.Mae(test, f.F), Accuracy.Rmse(test, f.F), Accuracy.Mape(test, f.F), Accuracy.Mase(test, f.F, train, period) }));
Chart.Lines("Ultimi 3 anni + previsioni a 12 mesi",
    new Line("osservato", y[84..]), new Line("seasonal naive", forecasts[2].F, Start: 24, Dashed: true), new Line("drift", forecasts[3].F, Start: 24, Dashed: true));

Metodo,MAE,RMSE,MAPE %,MASE (m=12)
Mean,36.5023,36.8154,21.5928,4.9649
Naive,6.3724,7.7112,3.7116,0.8667
Seasonal naive,6.9042,7.9845,4.0662,0.9391
Drift,5.458,6.3372,3.2146,0.7424


Ultimi 3 anni + previsioni a 12 mesi 140 150 160 170 180 0 10 20 30 40 <path d='M64 183.41L79.8 160.96L95.6 149.77L111.4 152.15L127.2 133.24L143 146.46L158.8 155.85L174.6 173.77L190.4 177.45L206.2 169.47L222 165.68L237.8 168.92L253.6 134.48L269.4 122.82L285.2 107.19L301 112.69L316.8 111.2L332.6 142.04L348.4 128.18L364.2 144.91L380 141.7L395.8 146.76L411.6 163.7L427.4 128.95L443.2 116.38L459 107.06L474.8 103.17L490.6 84.84L506.4 74.34L522.2 91.39L538 112.77L553.8 99.59L569.6 132.82L585.4 132.21L601.2 123.56L617 108.24' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/> osservato · x=0: 147 osservato · x=1: 154 osservato · x=2: 157 osservato · x=3: 156 osservato · x=4: 161 osservato · x=5: 158 osservato · x=6: 155 osservato · x=7: 150 osservato · x=8: 149 osservato · x=9: 151 osservato · x=10: 152 osservato · x=11: 151 osservato · x=12: 161 osservato · x=13: 164 osservato · x=14: 169 osservato · x=15: 167 osservato · x=16: 167 osservato · x=17: 159 osservato · x=18: 163 osservato · x=19: 158 osservato · x=20: 159 osservato · x=21: 158 osservato · x=22: 153 osservato · x=23: 163 osservato · x=24: 166 osservato · x=25: 169 osservato · x=26: 170 osservato · x=27: 175 osservato · x=28: 178 osservato · x=29: 173 osservato · x=30: 167 osservato · x=31: 171 osservato · x=32: 161 osservato · x=33: 162 osservato · x=34: 164 osservato · x=35: 168 seasonal naive · x=24: 161 seasonal naive · x=25: 164 seasonal naive · x=26: 169 seasonal naive · x=27: 167 seasonal naive · x=28: 167 seasonal naive · x=29: 159 seasonal naive · x=30: 163 seasonal naive · x=31: 158 seasonal naive · x=32: 159 seasonal naive · x=33: 158 seasonal naive · x=34: 153 seasonal naive · x=35: 163 drift · x=24: 163 drift · x=25: 164 drift · x=26: 164 drift · x=27: 165 drift · x=28: 165 drift · x=29: 166 drift · x=30: 167 drift · x=31: 167 drift · x=32: 168 drift · x=33: 168 drift · x=34: 169 drift · x=35: 170 drift seasonal naive osservato

## Autocovarianza, ACF e PACF
$$\hat\gamma(k) = \frac1n \sum_{t=1}^{n-k} (y_t-\bar y)(y_{t+k}-\bar y), \qquad \hat\rho(k)=\hat\gamma(k)/\hat\gamma(0).$$
Il divisore è $n$, non $n-k$: lo stimatore è distorto, ma la matrice di Toeplitz $\Gamma_{ij}=\hat\gamma(|i-j|)$ resta semidefinita positiva. È questa proprietà che garantisce $|\text{PACF}|\le 1$ e un AR stimato stazionario.

La **PACF** al lag $k$ è l'ultimo coefficiente $\phi_{kk}$ dell'AR($k$) che meglio approssima la serie. Si ottiene dalla ricorsione di **Levinson-Durbin**, che risolve le equazioni di Yule-Walker $\Gamma\phi=\gamma$ in $O(p^2)$:
$$\kappa_k = \frac{\gamma(k) - \sum_{j<k}\phi_{k-1,j}\,\gamma(k-j)}{v_{k-1}},\quad \phi_{k,j}=\phi_{k-1,j}-\kappa_k\phi_{k-1,k-j},\quad v_k=v_{k-1}(1-\kappa_k^2).$$
La banda $\pm 1.96/\sqrt n$ è il riferimento del rumore bianco. Firma tipica: un AR($p$) ha ACF che decade e PACF che si **taglia** dopo il lag $p$.

In [ ]:
// Yule (1927) modellò le macchie solari con un AR(2). φ = (1.6, −0.9) ha radici complesse: pseudo-ciclo.
var ar2 = Simulation.Arma<double>(new Random(1927), 600, 0, new[] { 1.6, -0.9 }, Array.Empty<double>(), 1);
var band = 1.96 / Math.Sqrt(ar2.Length);
Chart.Lines("AR(2) simulato, φ = (1.6, −0.9)", new Line("y", ar2[..200]));
Chart.Stems("ACF: oscillazione smorzata, periodo ≈ 11", SeriesStatistics.Acf(ar2, 30)[1..], band);
Chart.Stems("PACF: si taglia dopo il lag 2", SeriesStatistics.Pacf(ar2, 30), band);

AR(2) simulato, φ = (1.6, −0.9) -10 -5 0 5 10 0 50 100 150 200 <path d='M64 157.53L67.16 129.6L70.32 107.35L73.48 97.38L76.64 82.88L79.8 84.76L82.96 99.2L86.12 136.14L89.28 175.1L92.44 186.45L95.6 175.25L98.76 149.09L101.92 119.79L105.08 110.18L108.24 101.9L111.4 107.3L114.56 112.13L117.72 122.14L120.88 130.49L124.04 135.87L127.2 138.81L130.36 136.5L133.52 133.53L136.68 118.5L139.84 101.73L143 86.69L146.16 73.27L149.32 68.09L152.48 88.18L155.64 131.31L158.8 164.03L161.96 162.35L165.12 151.77L168.28 136.4L171.44 113.15L174.6 104.2L177.76 105.21L180.92 100.78L184.08 109.85L187.24 116.43L190.4 125.8L193.56 130.5L196.72 129.08L199.88 156.12L203.04 167.32L206.2 171.08L209.36 145.05L212.52 115.13L215.68 96.2L218.84 75.04L222 72.8L225.16 94.15L228.32 109.8L231.48 112.27L234.64 120.18L237.8 126.92L240.96 118.51L244.12 108.21L247.28 102.66L250.44 97.28L253.6 100.7L256.76 113.2L259.92 116.86L263.08 120.58L266.24 132.34L269.4 143.07L272.56 154.33L275.72 162.97L278.88 165.46L282.04 160.26L285.2 123.68L288.36 99.89L291.52 72.83L294.68 63.67L297.84 69.96L301 99.47L304.16 141.19L307.32 174.26L310.48 195.3L313.64 185.25L316.8 161.63L319.96 141.62L323.12 115.69L326.28 99.62L329.44 98.98L332.6 117.02L335.76 140.22L338.92 162.1L342.08 180.15L345.24 165.96L348.4 136.78L351.56 114.02L354.72 90.83L357.88 74.69L361.04 78.64L364.2 81.68L367.36 113.11L370.52 146.43L373.68 166.05L376.84 172.25L380 167L383.16 143.22L386.32 122.49L389.48 109.57L392.64 97.16L395.8 87.97L398.96 104.02L402.12 135.42L405.28 166.29L408.44 178.97L411.6 173.13L414.76 157.62L417.92 130.46L421.08 96.97L424.24 72.03L427.4 70.03L430.56 83.11L433.72 98.12L436.88 112.7L440.04 134.3L443.2 145.94L446.36 144.85L449.52 131.87L452.68 118.58L455.84 120.92L459 116.75L462.16 123.06L465.32 136.42L468.48 144.78L471.64 156.14L474.8 166.66L477.96 166.33L481.12 155.33L484.28 142.69L487.44 135.27L490.6 125.89L493.76 125.73L496.92 127.82L500.08 133.75L503.24 135.86L506.4 147.02L509.56 152.24L512.72 142.49L515.88 129.48L519.04 102.92L522.2 102.98L525.36 114.54L528.52 145.63L531.68 170.58L534.84 197.13L538 206.69L541.16 187.94L544.32 153.24L547.48 119.23L550.64 87.31L553.8 71.45L556.96 76.58L560.12 95.83L563.28 127.97L566.44 150.26L569.6 154.84L572.76 134.73L575.92 127.56L579.08 116.91L582.24 115.47L585.4 117.31L588.56 129.61L591.72 152.54L594.88 159.61L598.04 159.17L601.2 157.46L604.36 142.84L607.52 117.51L610.68 98.41L613.84 96.68L617 104.69L620.16 114.67L623.32 129.46L626.48 141.33L629.64 146.62L632.8 155.49L635.96 168.81L639.12 177.13L642.28 176.71L645.44 169.01L648.6 163.87L651.76 150.26L654.92 120.02L658.08 88.23L661.24 74.34L664.4 63.88L667.56 65.18L670.72 93.2L673.88 136.33L677.04 179.5L680.2 194.03L683.36 181.27L686.52 152.66L689.68 113.64L692.84 105.13' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/> y · x=0: -3.522 y · x=1: -0.074 y · x=2: 2.673 y · x=3: 3.903 y · x=4: 5.694 y · x=5: 5.462 y · x=6: 3.679 y · x=7: -0.881 y · x=8: -5.691 y · x=9: -7.093 y · x=10: -5.71 y · x=11: -2.48 y · x=12: 1.137 y · x=13: 2.324 y · x=14: 3.345 y · x=15: 2.68 y · x=16: 2.082 y · x=17: 0.846 y · x=18: -0.184 y · x=19: -0.848 y · x=20: -1.211 y · x=21: -0.926 y · x=22: -0.559 y · x=23: 1.296 y · x=24: 3.367 y · x=25: 5.223 y · x=26: 6.88 y · x=27: 7.52 y · x=28: 5.04 y · x=29: -0.285 y · x=30: -4.324 y · x=31: -4.117 y · x=32: -2.812 y · x=33: -0.913 y · x=34: 1.957 y · x=35: 3.061 y · x=36: 2.937 y · x=37: 3.484 y · x=38: 2.365 y · x=39: 1.552 y · x=40: 0.395 y · x=41: -0.186 y · x=42: -0.01 y · x=43: -3.348 y · x=44: -4.731 y · x=45: -5.195 y · x=46: -1.981 y · x=47: 1.712 y · x=48: 4.049 y · x=49: 6.662 y · x=50: 6.938 y · x=51: 4.303 y · x=52: 2.37 y · x=53: 2.066 y · x=54: 1.089 y · x=55: 0.256 y · x=56: 1.295 y · x=57: 2.567 y · x=58: 3.252 y · x=59: 3.915 y · x=60: 3.494 y · x=61: 1.951 y · x=62: 1.498 y · x=63: 1.04 y · x=64: -0.413 y · x=65: -1.738 y · x=66: -3.128 y · x=67: -4.193 y · x=68: -4.502 y · x=69: -3.86 y · x=70: 0.657 y · x=71: 3.594 y · x

ACF: oscillazione smorzata, periodo ≈ 11 -1 -0.5 0 0.5 1 10 20 30 Lag ACF: oscillazione smorzata, periodo ≈ 11 · x=1: 0.833 ACF: oscillazione smorzata, periodo ≈ 11 · x=2: 0.428 ACF: oscillazione smorzata, periodo ≈ 11 · x=3: -0.058 ACF: oscillazione smorzata, periodo ≈ 11 · x=4: -0.464 ACF: oscillazione smorzata, periodo ≈ 11 · x=5: -0.664 ACF: oscillazione smorzata, periodo ≈ 11 · x=6: -0.62 ACF: oscillazione smorzata, periodo ≈ 11 · x=7: -0.376 ACF: oscillazione smorzata, periodo ≈ 11 · x=8: -0.042 ACF: oscillazione smorzata, periodo ≈ 11 · x=9: 0.26 ACF: oscillazione smorzata, periodo ≈ 11 · x=10: 0.436 ACF: oscillazione smorzata, periodo ≈ 11 · x=11: 0.44 ACF: oscillazione smorzata, periodo ≈ 11 · x=12: 0.298 ACF: oscillazione smorzata, periodo ≈ 11 · x=13: 0.077 ACF: oscillazione smorzata, periodo ≈ 11 · x=14: -0.139 ACF: oscillazione smorzata, periodo ≈ 11 · x=15: -0.282 ACF: oscillazione smorzata, periodo ≈ 11 · x=16: -0.316 ACF: oscillazione smorzata, periodo ≈ 11 · x=17: -0.241 ACF: oscillazione smorzata, periodo ≈ 11 · x=18: -0.097 ACF: oscillazione smorzata, periodo ≈ 11 · x=19: 0.058 ACF: oscillazione smorzata, periodo ≈ 11 · x=20: 0.169 ACF: oscillazione smorzata, periodo ≈ 11 · x=21: 0.208 ACF: oscillazione smorzata, periodo ≈ 11 · x=22: 0.172 ACF: oscillazione smorzata, periodo ≈ 11 · x=23: 0.087 ACF: oscillazione smorzata, periodo ≈ 11 · x=24: -0.009 ACF: oscillazione smorzata, periodo ≈ 11 · x=25: -0.079 ACF: oscillazione smorzata, periodo ≈ 11 · x=26: -0.106 ACF: oscillazione smorzata, periodo ≈ 11 · x=27: -0.091 ACF: oscillazione smorzata, periodo ≈ 11 · x=28: -0.048 ACF: oscillazione smorzata, periodo ≈ 11 · x=29: -0.001 ACF: oscillazione smorzata, periodo ≈ 11 · x=30: 0.031 +band · x=0.5: 0.08 +band · x=30.5: 0.08 -band · x=0.5: -0.08 -band · x=30.5: -0.08 -band +band ACF: oscillazione smorzata, periodo ≈ 11

PACF: si taglia dopo il lag 2 -1 -0.5 0 0.5 1 10 20 30 Lag PACF: si taglia dopo il lag 2 · x=1: 0.833 PACF: si taglia dopo il lag 2 · x=2: -0.869 PACF: si taglia dopo il lag 2 · x=3: -0.013 PACF: si taglia dopo il lag 2 · x=4: 0.008 PACF: si taglia dopo il lag 2 · x=5: 0.106 PACF: si taglia dopo il lag 2 · x=6: -0.01 PACF: si taglia dopo il lag 2 · x=7: 0.007 PACF: si taglia dopo il lag 2 · x=8: -0.057 PACF: si taglia dopo il lag 2 · x=9: 0.007 PACF: si taglia dopo il lag 2 · x=10: 0.024 PACF: si taglia dopo il lag 2 · x=11: -0.048 PACF: si taglia dopo il lag 2 · x=12: 0.045 PACF: si taglia dopo il lag 2 · x=13: -0.003 PACF: si taglia dopo il lag 2 · x=14: -0.013 PACF: si taglia dopo il lag 2 · x=15: -0.038 PACF: si taglia dopo il lag 2 · x=16: -0.002 PACF: si taglia dopo il lag 2 · x=17: 0.051 PACF: si taglia dopo il lag 2 · x=18: -0.022 PACF: si taglia dopo il lag 2 · x=19: -0.032 PACF: si taglia dopo il lag 2 · x=20: -0.022 PACF: si taglia dopo il lag 2 · x=21: 0.034 PACF: si taglia dopo il lag 2 · x=22: 0.003 PACF: si taglia dopo il lag 2 · x=23: 0.028 PACF: si taglia dopo il lag 2 · x=24: 0.008 PACF: si taglia dopo il lag 2 · x=25: 0.032 PACF: si taglia dopo il lag 2 · x=26: -0.026 PACF: si taglia dopo il lag 2 · x=27: -0.011 PACF: si taglia dopo il lag 2 · x=28: 4.5e-4 PACF: si taglia dopo il lag 2 · x=29: -0.01 PACF: si taglia dopo il lag 2 · x=30: 0.01 +band · x=0.5: 0.08 +band · x=30.5: 0.08 -band · x=0.5: -0.08 -band · x=30.5: -0.08 -band +band PACF: si taglia dopo il lag 2

## Yule-Walker e il ciclo nascosto nelle radici
Per un AR(2) $y_t = \phi_1 y_{t-1} + \phi_2 y_{t-2} + \varepsilon_t$ il polinomio caratteristico $z^2-\phi_1 z-\phi_2$ ha radici complesse quando $\phi_1^2+4\phi_2<0$. Le radici sono $r e^{\pm i\theta}$ con $r=\sqrt{-\phi_2}$ e $\cos\theta = \phi_1/(2\sqrt{-\phi_2})$: l'ACF è una sinusoide smorzata di periodo $2\pi/\theta$.

Con $\phi=(1.6,-0.9)$: $\cos\theta = 1.6/(2\cdot 0.949) = 0.843$, quindi $\theta \approx 0.567$ e un periodo di circa **11.1** passi. È l'idea di Yule: un ciclo di circa 11 anni senza alcuna sinusoide deterministica, solo dinamica autoregressiva guidata da shock.

**Wold (1938)**: ogni processo stazionario a media nulla si scrive come $\sum_j \psi_j \varepsilon_{t-j}$ più una parte deterministica. È un teorema, non un modello: l'AR finito ne è un'approssimazione parsimoniosa.

In [ ]:
var yw = ArModel.FitYuleWalker(ar2, 2);
double Period(double phi1, double phi2) => 2 * Math.PI / Math.Acos(phi1 / (2 * Math.Sqrt(-phi2)));
Table.Show(new[] { "", "φ1", "φ2", "σ²", "periodo implicito" }, new[]
{
    new object[] { "vero", 1.6, -0.9, 1.0, Period(1.6, -0.9) },
    new object[] { "Yule-Walker", yw.Phi[0], yw.Phi[1], yw.InnovationVariance, Period(yw.Phi[0], yw.Phi[1]) },
});
Chart.Lines("Previsione AR(2): il ciclo si smorza verso la media", new Line("osservato", ar2[^60..]), new Line("previsione", yw.Forecast(ar2, 40), Start: 60, Dashed: true));

,φ1,φ2,σ²,periodo implicito
vero,1.6,-0.9,1,11.0727
Yule-Walker,1.5574,-0.8692,1.1724,10.7909


Previsione AR(2): il ciclo si smorza verso la media -7.5 -5 -2.5 0 2.5 5 7.5 0 20 40 60 80 100 <path d='M64 82.9L70.32 95.33L76.64 122.67L82.96 141.08L89.28 151.58L95.6 149.62L101.92 141.22L108.24 136.4L114.56 135.8L120.88 128.18L127.2 122.26L133.52 132.59L139.84 146.38L146.16 153.93L152.48 154.71L158.8 143.34L165.12 126.04L171.44 110.35L177.76 97.39L184.08 109.25L190.4 110.03L196.72 133.61L203.04 146.58L209.36 137.4L215.68 151.37L222 144.44L228.32 131.44L234.64 126.14L240.96 117.36L247.28 116.39L253.6 127.26L259.92 147.05L266.24 172.21L272.56 194.59L278.88 184.18L285.2 156.3L291.52 135.59L297.84 144.91L304.16 169.63L310.48 175.37L316.8 184.16L323.12 175.53L329.44 176.15L335.76 165.14L342.08 145.85L348.4 140.44L354.72 123.33L361.04 117.27L367.36 129.91L373.68 122.8L380 131.01L386.32 139.17L392.64 150.24L398.96 154.25L405.28 149.53L411.6 125.46L417.92 95.6L424.24 81.01L430.56 81.13L436.88 90.91' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/><path d='M443.2 114L449.52 141.45L455.84 164.13L462.16 175.59L468.48 173.72L474.8 160.86L481.12 142.44L487.44 124.94L493.76 113.7L500.08 111.4L506.4 117.59L512.72 129.23L519.04 141.98L525.36 151.72L531.68 155.8L538 153.69L544.32 146.86L550.64 138.06L556.96 130.28L563.28 125.82L569.6 125.64L575.92 129.23L582.24 134.98L588.56 140.82L594.88 144.91L601.2 146.2L607.52 144.66L613.84 141.15L620.16 137L626.48 133.6L632.8 131.92L639.12 132.24L645.44 134.22L651.76 137.01L658.08 139.64L664.4 141.31L670.72 141.62L677.04 140.66L683.36 138.89L689.68 136.97' fill='none' stroke='#eb6834' stroke-width='2' stroke-dasharray='6 4' stroke-linejoin='round' stroke-linecap='round'/> osservato · x=0: 5.74 osservato · x=1: 4.445 osservato · x=2: 1.597 osservato · x=3: -0.321 osservato · x=4: -1.414 osservato · x=5: -1.211 osservato · x=6: -0.335 osservato · x=7: 0.166 osservato · x=8: 0.229 osservato · x=9: 1.023 osservato · x=10: 1.639 osservato · x=11: 0.563 osservato · x=12: -0.873 osservato · x=13: -1.659 osservato · x=14: -1.74 osservato · x=15: -0.556 osservato · x=16: 1.245 osservato · x=17: 2.88 osservato · x=18: 4.23 osservato · x=19: 2.995 osservato · x=20: 2.913 osservato · x=21: 0.457 osservato · x=22: -0.894 osservato · x=23: 0.062 osservato · x=24: -1.393 osservato · x=25: -0.67 osservato · x=26: 0.684 osservato · x=27: 1.236 osservato · x=28: 2.15 osservato · x=29: 2.251 osservato · x=30: 1.119 osservato · x=31: -0.943 osservato · x=32: -3.564 osservato · x=33: -5.895 osservato · x=34: -4.811 osservato · x=35: -1.906 osservato · x=36: 0.251 osservato · x=37: -0.719 osservato · x=38: -3.294 osservato · x=39: -3.893 osservato · x=40: -4.809 osservato · x=41: -3.909 osservato · x=42: -3.974 osservato · x=43: -2.828 osservato · x=44: -0.817 osservato · x=45: -0.254 osservato · x=46: 1.528 osservato · x=47: 2.159 osservato · x=48: 0.843 osservato · x=49: 1.584 osservato · x=50: 0.728 osservato · x=51: -0.122 osservato · x=52: -1.275 osservato · x=53: -1.692 osservato · x=54: -1.201 osservato · x=55: 1.306 osservato · x=56: 4.417 osservato · x=57: 5.937 osservato · x=58: 5.924 osservato · x=59: 4.905 previsione · x=60: 2.5 previsione · x=61: -0.359 previsione · x=62: -2.722 previsione · x=63: -3.915 previsione · x=64: -3.721 previsione · x=65: -2.381 previsione · x=66: -0.463 previsione · x=67: 1.36 previsione · x=68: 2.531 previsione · x=69: 2.771 previsione · x=70: 2.126 previsione · x=71: 0.913 previsione · x=72: -0.415 previsione · x=73: -1.429 previsione · x=74: -1.854 previsione · x=75: -1.634 previsione · x=76: -0.923 previsione · x=77: -0.006 previsione · x=78: 0.804 previsione · x=79: 1.268 previsione · x=80: 1.287 previsione · x=81: 0.913 previsione · x=82: 0.314 previsione · x=83: -0.294 previsione · x=84: -0.719 previsione · x=85: -0.854 previsione · x=86: -0.694 previsione · x=87: -0.328 previsione · x=88: 0.104 previsione · x=89: 0.458 previsione · x=90: 0.633 previsione · x=91: 0.6 previsione · x=92: 0.394 previsione · x=93: 0.103 previsione · x=

## Controllo dei residui: Ljung-Box
$Q = n(n+2)\sum_{k=1}^{h}\hat\rho_k^2/(n-k) \sim \chi^2_{h-p}$ se i residui sono rumore bianco. Un p-value piccolo indica che il modello non ha catturato tutta la dinamica. Il p-value usa la gamma incompleta regolarizzata, implementata in `SpecialFunctions` (Lanczos + serie/frazione continua).

In [ ]:
Table.Show(new[] { "Modello", "σ² residua", "Ljung-Box Q(20)", "p-value" }, Enumerable.Range(1, 3).Select(p =>
{
    var model = ArimaModel.FitHannanRissanen(ar2, p, 0, 0);
    var residuals = model.Residuals(ar2)[p..];
    var lb = SeriesStatistics.LjungBox(residuals, 20, p);
    return new object[] { $"AR({p})", model.InnovationVariance, lb.Statistic, lb.PValue };
}));

Modello,σ² residua,Ljung-Box Q(20),p-value
AR(1),4.7721,1845.7512,0
AR(2),1.0703,15.0031,0.6618
AR(3),1.0728,14.7113,0.6163


## Esperimenti
- Portare φ2 a −0.5: le radici diventano più interne al cerchio unitario e il ciclo si smorza prima.
- φ = (0.5, 0.45): radici reali, nessun ciclo; quale firma hanno ACF e PACF?
- Applicare la seasonal naive a una serie senza stagionalità: la MASE resta sotto 1?